In [1]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

df = pd.read_csv("../cleaned_data/cleaned_city_day.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df[["City","Date","AQI"]].dropna()

cities = df["City"].unique()
forecast_results = []

for city in cities:
    city_df = df[df["City"] == city][["Date","AQI"]]
    city_df = city_df.groupby("Date").mean()
    city_df = city_df.asfreq("D")
    city_df["AQI"] = city_df["AQI"].interpolate()

    if len(city_df) < 200:
        continue

    try:
        model = ARIMA(city_df["AQI"], order=(5,1,0))
        model_fit = model.fit()

        forecast = model_fit.forecast(steps=365)

        for date, value in forecast.items():
            forecast_results.append([city, date, value])

    except:
        print("ARIMA failed for:", city)

arima_forecast_df = pd.DataFrame(forecast_results, columns=["City","Date","Forecast_AQI"])
arima_forecast_df.to_csv("../forecasts/arima_forecast_all_cities.csv", index=False)

print("ARIMA forecast saved!")

ARIMA forecast saved!
